# phonondft

Phonon dispersion of a structure relaxed with [CellRelaxDFT](https://nanohub.org/tools/cellrelaxdft): finite displacements with phonopy + Quantum ESPRESSO, and Born charges / epsilon_inf (DFPT) for the LO-TO correction of insulators.

There is **no cutoff convergence scan**: `ecutwfc` is an input with a guessed default. Check it for your pseudopotentials.

In [ ]:
DESCRIPTION = "Phonon dispersion (phonopy + Quantum ESPRESSO) of a cellrelaxdft-relaxed structure"

In [ ]:
%load_ext yamlmagic
import os
import numpy as np
from simtool import DB
import phonondft_lib as lib

In [ ]:
EXTRA_FILES = ["phonondft_lib.py"]

In [ ]:
%%yaml INPUTS

squid_id:
    description: squid of a finished cellrelaxdft run (copy it from that run)
    type: Text
    value: 'cellrelaxdft/r101/e31f06391363b332d66b0d3e0f3658d8a99b90b7'

ecutwfc:
    description: Wavefunction cutoff. No convergence scan is run; 80 Ry is a guess for norm-conserving pseudopotentials. 0 = reuse the relaxation's cutoff.
    type: Number
    units: Ry
    value: 80
    min: 0
    max: 400

ecutrho_dual:
    description: ecutrho / ecutwfc. 8 is safe for semicore states; 4 can give wrong force constants (seen for Na in NaCl).
    type: Number
    value: 8
    min: 4
    max: 16

kspacing:
    description: Monkhorst-Pack k-point spacing (2*pi included); the grid for each cell follows from it
    type: Number
    units: 1/Angstrom
    value: 0.25
    min: 0.05
    max: 1.0

supercell_length:
    description: Minimum supercell edge; supercell = n x conventional cell
    type: Number
    units: Angstrom
    value: 10.0
    min: 4.0
    max: 30.0

displacement:
    description: Finite-displacement amplitude
    type: Number
    units: bohr
    value: 0.02
    min: 0.005
    max: 0.1

metal:
    description: Smearing occupations and no Born charges (set True for metals)
    type: Boolean
    value: False

degauss:
    description: Marzari-Vanderbilt smearing width (metals only)
    type: Number
    units: Ry
    value: 0.02
    min: 0.001
    max: 0.1

compute_born:
    description: Born charges and epsilon_inf (ph.x) for the LO-TO correction (insulators)
    type: Boolean
    value: True

pseudo_dir:
    description: Directory with the UPF files. Empty = take them from the cellrelaxdft run.
    type: Text
    value: ''

plot_dos:
    description: Draw the phonon DOS next to the dispersion
    type: Boolean
    value: True

npoints:
    description: q-points per segment of the band path
    type: Integer
    value: 101
    min: 11
    max: 501

ncpus:
    description: Cores per QE job
    type: Integer
    value: 16
    min: 1
    max: 256

walltime:
    description: Wall time per QE job, in minutes
    type: Integer
    value: 120
    min: 5
    max: 1440

max_parallel_submits:
    description: QE jobs running at the same time
    type: Integer
    value: 8
    min: 1
    max: 50

submit_command:
    description: How each QE job is launched; {ncpus} {walltime} {exe} {input} are filled in
    type: Text
    value: 'submit -n {ncpus} -w {walltime} {exe} -in {input}'

pw_executable:
    description: pw.x name for the submit command (check the QE version installed on nanoHUB)
    type: Text
    value: 'pw.x'

ph_executable:
    description: ph.x name for the submit command
    type: Text
    value: 'ph.x'


In [ ]:
%%yaml OUTPUTS

spacegroup:
    description: Space group of the relaxed structure
    type: Text

ecutwfc:
    description: Wavefunction cutoff used
    type: Number
    units: Ry

ecutrho:
    description: Charge-density cutoff used
    type: Number
    units: Ry

cutoff_note:
    description: Where the cutoffs came from (given, relaxation, or guess)
    type: Text

supercell:
    description: Supercell multiples of the conventional cell
    type: List

natoms_supercell:
    description: Atoms in the supercell
    type: Integer

gamma_phonon_frequencies:
    description: Frequencies at Gamma (LO-TO split for polar insulators)
    type: Array
    units: 1/cm

min_frequency:
    description: Lowest frequency on the band path (negative = imaginary mode)
    type: Number
    units: 1/cm

band_structure:
    description: Band path distances, frequencies (1/cm) and segment labels
    type: Dict

dispersion_plot:
    description: Phonon dispersion (and DOS)
    type: Image

eps_inf:
    description: High-frequency dielectric tensor (-1 if not computed)
    type: Array

born_charges:
    description: Born effective charges per atom of the primitive cell (-1 if not computed)
    type: Array

band_gap:
    description: Kohn-Sham band gap from the Born scf (-1 if not computed)
    type: Number
    units: eV

phonopy_params:
    description: phonopy_params.yaml (structure, forces, Born charges) for further analysis
    type: File

warnings:
    description: Things to check before trusting the results
    type: List


In [ ]:
from simtool import getValidatedInputs

defaultInputs = getValidatedInputs(INPUTS)
if defaultInputs:
    globals().update(defaultInputs)

**** Computation is Done Below ****

In [ ]:
# Relaxed structure from the cellrelaxdft run
relax_outputs = lib.load_cellrelax(squid_id)
atoms, source = lib.relaxed_structure(relax_outputs)
name = lib.formula(atoms)
print(f'{name}: structure read from output "{source}", {len(atoms)} atoms')

In [ ]:
# Pseudopotentials: the input directory, or the ones the relaxation used
pdir = pseudo_dir or lib.pseudo_dir_from(relax_outputs)
if not pdir:
    raise RuntimeError('No pseudopotential directory found in the cellrelaxdft run; '
                       'set the pseudo_dir input')
pseudos = lib.find_pseudos(pdir, set(atoms.get_chemical_symbols()))
for el, p in pseudos.items():
    print(f"{el}: {p['file']} ({p['type']})")

In [ ]:
# Cutoffs: no scan. Given value, else the relaxation's, else a guess
ecutwfc_used, ecutrho_used, cutoff_note = lib.choose_cutoffs(ecutwfc, ecutrho_dual,
                                                              pseudos, relax_outputs)
print(cutoff_note)

In [ ]:
results = lib.run_phonons(
    atoms, pseudos, pdir, ecutwfc_used, ecutrho_used, name=name,
    metal=metal, degauss=degauss, kspacing=kspacing,
    supercell_length=supercell_length, displacement=displacement,
    compute_born=compute_born, npoints=npoints, dos=plot_dos,
    command=submit_command, pw_exe=pw_executable, ph_exe=ph_executable,
    ncpus=ncpus, walltime=walltime, max_parallel=max_parallel_submits,
    workdir='phonon_run')

In [ ]:
db = DB(OUTPUTS)
missing = np.array([-1.0])
db.save('spacegroup', results['spacegroup'])
db.save('ecutwfc', ecutwfc_used)
db.save('ecutrho', ecutrho_used)
db.save('cutoff_note', cutoff_note)
db.save('supercell', results['supercell'])
db.save('natoms_supercell', results['natoms_supercell'])
db.save('gamma_phonon_frequencies', np.array(results['gamma_cm1']))
db.save('min_frequency', results['min_cm1'])
db.save('band_structure', results['band_structure'])
db.save('dispersion_plot', file=results['plot'], display=True)
db.save('eps_inf', np.array(results['eps_inf']) if results['eps_inf'] is not None else missing)
db.save('born_charges', np.array(results['born_charges']) if results['born_charges'] is not None else missing)
db.save('band_gap', results['band_gap_ev'] if results['band_gap_ev'] is not None else -1.0)
db.save('phonopy_params', file=results['phonopy_params'])
db.save('warnings', results['warnings'], display=True)